In [2]:
import numpy as np
import pandas as pd

In [7]:

Math = "/content/Math 534 Baseball Project.xlsx"
prob_file = "/content/MLB_2023_2026_5Teams_Player_Probabilities.xlsx"


stats = pd.read_excel(Math)
probs = pd.read_excel(prob_file)

# 2. Filter to 2025 Dodgers
team = "Dodgers"
season = 2025

stats["Team"] = stats["Team"].replace({"Angles": "Angels"})

s = stats[
    (stats["Team"] == team) &
    (stats["Season"] == season)
].copy()

p = probs[
    (probs["team"] == team) &
    (probs["season"] == season)
].copy()

# Match names across files
s["key"] = s["Player"].str.strip().str.casefold()
p["key"] = p["player"].str.strip().str.casefold()

# Remove duplicate player rows
s = s.sort_values("PA", ascending=False).drop_duplicates("key")
p = p.drop_duplicates("key")

# Join statistics with outcome probabilities
data = s.merge(
    p.drop(columns=["player"]),
    on="key",
    how="inner",
    suffixes=("_stats", "_prob")
)

# Choose nine hitters with the most PA
players = data.sort_values("PA_stats", ascending=False).head(9).copy()

assert len(players) == 9, "Fewer than 9 players matched. Check player names."

# 3. Construct traditional lineup
remaining = players.copy()
traditional = []

rules = [
    ("OBP", False),
    ("OBP", False),
    ("OPS", False),
    ("SLG", False),
    ("OPS", False)
]

for metric, ascending in rules:
    chosen = remaining.sort_values(
        metric, ascending=ascending
    ).iloc[0]

    traditional.append(chosen)
    remaining = remaining.drop(chosen.name)

# Positions 6-9: remaining hitters, ordered by OPS (descending)
bottom_four = remaining.sort_values("OPS", ascending=False)

traditional = pd.DataFrame(traditional + [
    row for _, row in bottom_four.iterrows()
]).reset_index(drop=True)

# 4. OPS strategy: reorder same top five
ops_lineup = pd.concat([
    traditional.iloc[:5].sort_values("OPS", ascending=False),
    traditional.iloc[5:]
], ignore_index=True)

print("TRADITIONAL LINEUP")
print(traditional[["Player", "OBP", "OPS", "SLG"]].to_string(index=False))

print("\nOPS DESCENDING LINEUP")
print(ops_lineup[["Player", "OBP", "OPS", "SLG"]].to_string(index=False))


TRADITIONAL LINEUP
           Player   OBP   OPS   SLG
    Shohei Ohtani 0.392 1.014 0.622
        Max Muncy 0.376 0.846 0.470
  Freddie Freeman 0.367 0.869 0.502
Teoscar Hernández 0.339 0.840 0.501
     Mookie Betts 0.372 0.863 0.491
       Andy Pages 0.313 0.774 0.461
       Will Smith 0.327 0.760 0.433
      Tommy Edman 0.274 0.656 0.382
 Michael Conforto 0.305 0.638 0.333

OPS DESCENDING LINEUP
           Player   OBP   OPS   SLG
    Shohei Ohtani 0.392 1.014 0.622
  Freddie Freeman 0.367 0.869 0.502
     Mookie Betts 0.372 0.863 0.491
        Max Muncy 0.376 0.846 0.470
Teoscar Hernández 0.339 0.840 0.501
       Andy Pages 0.313 0.774 0.461
       Will Smith 0.327 0.760 0.433
      Tommy Edman 0.274 0.656 0.382
 Michael Conforto 0.305 0.638 0.333


In [9]:
import numpy as np

# 5. Monte Carlo simulation
outcomes = ["Single", "Double", "Triple", "Home Run",
            "Walk/HBP", "Error/Interference", "Out"]

def simulate_game(lineup, rng, innings=9):
    cum = lineup[outcomes].to_numpy(dtype=float).cumsum(axis=1)  # cumulative probs per batter
    batter, runs = 0, 0

    for _ in range(innings):
        outs = 0
        b1 = b2 = b3 = 0                      # 1 = runner on base

        while outs < 3:
            k = min(np.searchsorted(cum[batter], rng.random()), 6)  # outcome of this PA

            if k == 0:                        # single
                runs += b3 + b2
                b3, b2, b1 = 0, b1, 1
            elif k == 1:                      # double
                runs += b3 + b2
                b3, b2, b1 = b1, 1, 0
            elif k == 2:                      # triple
                runs += b1 + b2 + b3
                b3, b2, b1 = 1, 0, 0
            elif k == 3:                      # home run
                runs += b1 + b2 + b3 + 1
                b1 = b2 = b3 = 0
            elif k in (4, 5):                 # walk/HBP or error (forced advances only)
                if b1 and b2 and b3:
                    runs += 1
                b3 = 1 if (b3 or (b1 and b2)) else 0
                b2 = 1 if (b2 or b1) else 0
                b1 = 1
            else:                             # out
                outs += 1

            batter = (batter + 1) % 9
    return runs

n_games = 20000

def run_simulation(lineup, seed=534):
    rng = np.random.default_rng(seed)
    return np.array([simulate_game(lineup, rng) for _ in range(n_games)])

runs_trad = run_simulation(traditional)
runs_ops = run_simulation(ops_lineup)

# 6. Results
print(f"Traditional: {runs_trad.mean():.3f} runs/game")
print(f"OPS:         {runs_ops.mean():.3f} runs/game")



Traditional: 5.296 runs/game
OPS:         5.270 runs/game
